# **Deep Research**

Agentic use case of research using the Tavily tool.

## Setup

In [1]:
from agents import Agent, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from tavily import TavilyClient

load_dotenv()

True

In [2]:
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

response = client.search(query="Most recent news on Agentic AI",
                         search_depth="advanced",
                         max_results=3)

print(response)


{'query': 'Most recent news on Agentic AI', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://finance.yahoo.com/sectors/technology/articles/agentic-ai-goes-mainstream-enterprise-000000271.html', 'title': 'Agentic AI Goes Mainstream in the Enterprise, but 94% Raise Concern About Sprawl, OutSystems Research Finds', 'content': 'SINGAPORE, April 13, 2026 /PRNewswire/ -- OutSystems, a leading AI development platform, today released its global 2026 State of AI Development report, revealing that enterprises have moved decisively from AI experimentation to execution. Nearly every organization surveyed, 96%, is already using AI agents in some capacity, and 97% are exploring system-wide agentic AI strategies. The findings signal a clear shift from pilots to production as businesses embed AI into mission-critical operations.\n\nThis shift is increasingly visible in APAC, where markets such as India are already reporting advanced levels of agentic AI capabilit

## 1. Search Agent - Adding the Search Tool

In [3]:
@function_tool
def tavily_search(query: str) -> str:
    """
    Perform a web search using Tavily and return a concise summary of results.
    """
    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=5
    )
    
    results = []
    for r in response["results"]:
        results.append(f"{r['title']}\n{r['content']}")
    
    return "\n\n".join(results)

In [4]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[tavily_search],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)


In [5]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))


In 2025, several AI agent frameworks are emerging, significantly enhancing automation and decision-making in machine learning operations. Key frameworks include **AutoGen**, which excels in orchestrating multi-agent collaboration through a no-code interface, making it suitable for enterprises needing dynamic workflows. Other notable frameworks are **LangChain** and **LangGraph**, providing modular tools for complex workflows and strong state management features. **LlamaIndex** and **Haystack** cater to data-driven applications, while conversational AI frameworks like **Rasa** and **Botpress** remain vital for developing chatbots.

User-friendliness and integration capabilities are essential selection criteria for these frameworks. Tools like **n8n** and **Flowise** stand out for their visual interfaces geared toward non-coders, while those requiring extensive coding and integration might opt for more advanced setups like **Semantic Kernel**. Open-source alternatives such as **CrewAI** appeal to developers looking for flexibility, supporting role-based task partitioning among agents. Overall, the landscape reflects a trend toward frameworks that balance complexity with accessibility, catering to a wide range of enterprise needs and expertise levels.

### **As always, take a look at the trace**

https://platform.openai.com/traces

## 2. Planner Agent - Including Structured Outputs

In [6]:
HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)


In [7]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)


searches=[WebSearchItem(reason='To find the most recent AI agent frameworks released or updated in 2025.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To identify emerging trends and features in AI agent frameworks in 2025.', query='trends in AI agent frameworks 2025'), WebSearchItem(reason='To gather information on new tools and libraries specifically designed for AI agents.', query='new AI agent development tools 2025'), WebSearchItem(reason='To see expert opinions and reviews on the best AI agent frameworks in 2025.', query='best AI agent frameworks reviews 2025'), WebSearchItem(reason='To check for any conferences or publications related to AI agents in 2025.', query='AI agent frameworks conferences 2025')]


In [8]:
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='To find the most recent AI agent frameworks released or updated in 2025.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To identify emerging trends and features in AI agent frameworks in 2025.', query='trends in AI agent frameworks 2025'), WebSearchItem(reason='To gather information on new tools and libraries specifically designed for AI agents.', query='new AI agent development tools 2025'), WebSearchItem(reason='To see expert opinions and reviews on the best AI agent frameworks in 2025.', query='best AI agent frameworks reviews 2025'), WebSearchItem(reason='To check for any conferences or publications related to AI agents in 2025.', query='AI agent frameworks conferences 2025')])

## 3. Email Agent

In [9]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("plvital422@gmail.com")
    to_email = To("pedro.vital@ufms.br")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"


In [10]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

## 4. Writer Agent

In [11]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)


## 5. Functions to plan and execute the search, using the Planner Agent and Search Agent

In [12]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}") # result.final_output is an object of WebSearchPlan
    print(f"Will perform {len(result.final_output.searches)} searches") # result.final_output.searches is list[WebSearchItem]
    return result.final_output 

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output # simply markdown
    
async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches] # search_plan.searches is list[WebSearchItem]
    # create_task wraps each coroutine in a Task and schedules it immediately
    # on the event loop, allowing them to run concurrently before gather awaits them
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


`create_task` is optional here

You could simplify to:

```python
results = await asyncio.gather(*(search(item) for item in search_plan.searches))
```

**Why this works:**
`asyncio.gather` already schedules coroutines concurrently. Explicit `create_task` is only necessary if:

* You want to **track or manage tasks individually**
* You want tasks to start running **before gather is awaited**

---

When you call:

```python
tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
```

each `search(item)` coroutine is:

1. Wrapped into a `Task`
2. **Immediately scheduled on the event loop**

So they become *runnable* right away.

### Subtle but critical nuance

They don’t literally “run” at that exact line — because Python is still executing your current synchronous code. The event loop only gets a chance to actually *advance* those tasks when control is yielded back to it.

That happens here:

```python
results = await asyncio.gather(*tasks)
```

At that point:

* The event loop resumes control
* Any already-scheduled tasks (your `create_task` ones) can start progressing
* `gather` just waits for their completion

### So what’s the practical effect?

* With `create_task`: tasks are **ready and queued** before `await gather`
* Without it: tasks are **created and scheduled by `gather` itself at await time**

In your specific code, the difference is minimal because there’s no `await` between task creation and `gather`.

### When it *actually* changes behavior

If you had something like:

```python
tasks = [asyncio.create_task(search(item)) for item in items]

await asyncio.sleep(1)  # yield control to event loop

results = await asyncio.gather(*tasks)
```

Now:

* The tasks **start running during that 1 second**
* By the time you call `gather`, some may already be partially or fully done

### Bottom line

* ✔ `create_task` schedules coroutines immediately
* ✔ They *begin execution* as soon as the event loop gets control
* ✔ In your code, that effectively happens at `await gather(...)`






## 6. The next 2 functions write a report and email it


In [13]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report



## 7. Showtime!


In [14]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


https://platform.openai.com/logs?api=traces

## 8. Research Manager class

The class will be used as a wrapper for the callback function that will be passed to the Gradio app.

In [ ]:
class ResearchManager:

    async def run(self, query: str):
        """ Run the deep research process, yielding the status updates and the final report"""
        trace_id = gen_trace_id()
        with trace("Research trace", trace_id=trace_id):
            print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}")
            yield f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"
            print("Starting research...")
            search_plan = await self.plan_searches(query)
            yield "Searches planned, starting to search..."     
            search_results = await self.perform_searches(search_plan)
            yield "Searches complete, writing report..."
            report = await self.write_report(query, search_results)
            yield "Report written, sending email..."
            await self.send_email(report)
            yield "Email sent, research complete"
            yield report.markdown_report
        

    async def plan_searches(self, query: str) -> WebSearchPlan:
        """ Plan the searches to perform for the query """
        print("Planning searches...")
        result = await Runner.run(
            planner_agent,
            f"Query: {query}",
        )
        print(f"Will perform {len(result.final_output.searches)} searches")
        return result.final_output_as(WebSearchPlan)

    async def perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        """ Perform the searches to perform for the query """
        print("Searching...")
        num_completed = 0
        tasks = [asyncio.create_task(self.search(item)) for item in search_plan.searches]
        results = []
        for task in asyncio.as_completed(tasks):
            result = await task
            if result is not None:
                results.append(result)
            num_completed += 1
            print(f"Searching... {num_completed}/{len(tasks)} completed")
        print("Finished searching")
        return results

    async def search(self, item: WebSearchItem) -> str | None:
        """ Perform a search for the query """
        input = f"Search term: {item.query}\nReason for searching: {item.reason}"
        try:
            result = await Runner.run(
                search_agent,
                input,
            )
            return str(result.final_output)
        except Exception:
            return None

    async def write_report(self, query: str, search_results: list[str]) -> ReportData:
        """ Write the report for the query """
        print("Thinking about report...")
        input = f"Original query: {query}\nSummarized search results: {search_results}"
        result = await Runner.run(
            writer_agent,
            input,
        )

        print("Finished writing report")
        return result.final_output_as(ReportData)
    
    async def send_email(self, report: ReportData) -> None:
        print("Writing email...")
        await Runner.run(
            email_agent,
            report.markdown_report,
        )
        print("Email sent")
        return report

### **The perform_searches coroutine**

This function executes **multiple independent web searches concurrently** using Python’s `asyncio` framework. Instead of running searches one after another, it schedules all of them at once and processes results **as soon as each search finishes**.

This pattern is ideal for:

* Network-bound tasks (HTTP requests, APIs)
* Independent operations
* Improving throughput without threads or processes

---

#### Creating Tasks

```python
tasks = [asyncio.create_task(self.search(item)) for item in search_plan.searches]
```

This is the **most important line** in the function.

**What is happening here?**

* `self.search(item)`
  This is an *async function call*, which produces a **coroutine object**.

* `asyncio.create_task(...)`
  Converts the coroutine into a **Task** and immediately schedules it on the event loop.

> the coroutines only start executing when control is yielded back to the event loop (e.g., at an ```await```).

That means:

* All searches are started almost at the same time.
* They can overlap while waiting for network I/O.
* None of them block the others.

At the end of this line:

* You have a list of scheduled tasks.

---

#### Processing Tasks as They Complete

```python
for task in asyncio.as_completed(tasks):
```

This is a key concurrency primitive.

**What `asyncio.as_completed` does**

* Takes an iterable of `Task` objects
* Returns an iterator that yields tasks **in the order they finish**, not the order they were started

Important implications:

* Fast searches are processed first
* Slow searches do not block the entire loop
* You get results as soon as they are available

This is different from `asyncio.gather`, which waits for *all* tasks to finish before returning.

---

#### Awaiting Each Completed Task

```python
result = await task
```

At this point:

* The task has already completed (or is about to complete)
* `await task` retrieves its return value
* If the task raised an exception, it would be raised here

Because tasks were scheduled earlier, this `await`:

* Does **not** restart the task
* Only collects its result

---

#### Handling Results Safely

```python
if result is not None:
    results.append(result)
```

Your `search` method returns:

* A string on success
* `None` on failure (caught exception)

This check:

* Filters out failed searches
* Allows the pipeline to continue even if some searches fail

This is a **fault-tolerant design**.

---


#### Execution Timeline (Conceptual)

1. All searches are scheduled immediately
2. Event loop switches between them while they wait on I/O
3. Tasks finish in unpredictable order
4. Results are processed as soon as each finishes
5. Final list is returned


#### Compared to `asyncio.gather`

| Approach       | Behavior                                         |
| -------------- | ------------------------------------------------ |
| `gather`       | Waits for all tasks, returns all results at once |
| `as_completed` | Processes tasks as soon as they finish           |

You chose `as_completed` because:

* You want progress updates
* You want resilience to slow or failing tasks
* You want better observability

